<div class="alert alert-block alert-info">
    

This cell includes the time and cost for running this notebook using the environment information below. Increasing the number of workers will allow the job to complete more quickly.
    
<li> Main node: n2-standard-16, 500GB Disk</li>

<li> Workers (2/10): n2-standard-8, 500GB Disk</li>

<li> Time and Cost: \$4.72/h, ~30 min, \$2.35</li>
    
<li> The number of workers can be increased to shorten the time needed to run the notebook.</li> 
    
Please note that some lines of code (additional filtering and export) were commented out to reduce the time and cost of running the notebook. 
<li> The user can choose to only uncomment and run certain export steps (rather than exporting all test files).

</div>

In [ ]:
!git config --global user.email "shubhankar.londhe@gmail.com"
!git config --global user.name "ShubhankarLondhe"

# Notebook Setup

In [ ]:
!pip install polars
!pip install plotnine

In [ ]:
import os
import glob
import subprocess

import polars as pl
from scipy.stats import false_discovery_control
from statsmodels.stats.multitest import multipletests
from plotnine import *

from datetime import datetime
# Record time needed to run the notebook
start = datetime.now()

In [ ]:
# Initialize Hail without setting default reference
import hail as hl
hl.init()

# Set the default reference genome after initialization
hl.default_reference('GRCh38')

In [ ]:
#Path functions to make it easier to load data tables
def get_ht_path(type, ancestry, pheno):
    return f'gs://vwb-aou-allxall/v8/ht/{type}/{ancestry.upper()}/phenotype_{pheno}_{type}_results.ht'

def get_mt_path(ancestry, type):
    return f'gs://vwb-aou-allxall/v8/mt/{ancestry.upper()}_{type}_results.mt'

# Get gene trait associations from Hail MT

In [ ]:
%%time
# Test with ONE ancestry only, no loop
mt = hl.read_matrix_table(get_mt_path(ancestry='META', type="gene"))
ht = mt.entries()
ht.count()  # <-- just count rows first, fast sanity check

In [ ]:
mt.aggregate_cols(hl.agg.counter(mt.category))

In [ ]:
mt = hl.read_matrix_table(get_mt_path(ancestry='META', type="gene"))
print(list(mt.entry))

In [ ]:
%%time

# categories = hl.literal({'mcc2_phecodex', 'r_drug', 'physical_measurement', 'lab_measurement'})
categories = hl.literal({'physical_measurement', 'lab_measurement'})

# Column filter: phenotype categories
mt_sub = mt.filter_cols(categories.contains(mt.category))

# Row filter: annotation + MAF threshold
mt_sub = mt_sub.filter_rows(
    (mt_sub.annotation == "pLoF") &
    (mt_sub.max_MAF == 0.001)
)

mt_sub.entries().show()

In [ ]:
%%time

# Now flatten — much smaller cross product
ht = mt_sub.entries()

# Entry filter: Pvalue (this is an entry field, can only be filtered after entries())
ht_filtered = ht.filter(
    (hl.is_defined(ht.META_Pvalue_Burden)) & (ht.META_Pvalue_Burden < 6.7 * 1e-7)
    # (hl.is_defined(ht.META_Pvalue_SKATO)) & (ht.META_Pvalue_SKATO < 6.7 * 1e-7)
)

ht_filtered.show()

In [ ]:
ht_filtered.count()

In [ ]:
%%time

# Save hail table as csv
# ht_filtered.select('description', 'ancestries', 'META_Stats_Burden', 'META_Pvalue_Burden').export('<AOU_GYM_DATA_ROOT>/rvat_gene_results/pLoF_Burden_sig_META_MAF1e-3.tsv')


In [ ]:
%%time
gene_ids = ht_filtered.aggregate(hl.agg.collect_as_set(ht_filtered.gene_id))
phenonames = ht_filtered.aggregate(hl.agg.collect_as_set(ht_filtered.phenoname))

print(f"Unique genes: {len(gene_ids)}")
print(f"Unique phenotypes: {len(phenonames)}")

In [ ]:
ht_filtered.aggregate(hl.agg.collect_as_set(ht_filtered.category))

# Explore All by All results

In [ ]:
#This cell lists all Hail MatrixTable results
!gcloud storage ls "gs://vwb-aou-allxall/v8/mt/"

In [ ]:
# The results table used for the example will be the GWAS results for ACAF variants for the SAS genetic ancestry group:
mt_exome = hl.read_matrix_table(get_mt_path(ancestry = "META", type = "exome"))

#Print file path of the results table 
get_mt_path(ancestry = "META", type = "exome")

In [ ]:
print(mt_exome.row_key)
print([f for f in mt_exome.row])      # row field names
print([f for f in mt_exome.entry])    # entry field names

In [ ]:
# mt_exome.describe()

# Write sum stats as MatrixTable to gcloud for all (gene, pheno) pairs across all ancestries

In [ ]:
# # Your lists
# my_genes = ['ENSG00000130164', 'ENSG00000169174']  # e.g. LDLR, PCSK9 ENSG IDs
# my_phenotypes = ["pheno1", "pheno2"]            # your phenoname list

# # Subset rows to your genes, cols to your phenotypes
# mt_sub = mt_exome.filter_rows(hl.literal(set(my_genes)).contains(mt_exome.gene_id))
# mt_sub = mt_sub.filter_cols(hl.literal(set(my_phenotypes)).contains(mt_sub.phenoname))

# mt_sub.show(n_rows=5, n_cols=5)

In [ ]:
%%time
import hail as hl

# OUT_PATH = '<AOU_OUTPUT_ROOT>/ukbgym_exome_sumstats'
OUT_PATH = os.environ["AOU_GYM_DATA_ROOT"].rstrip("/") + "/allxall_exome_sumstats"

print("Number of unique genes:", len(gene_ids))
print("Number of unique phenonames:", len(phenonames))

mt_meta = hl.read_matrix_table(get_mt_path(ancestry='META', type="exome"))  # variant-level path

# QC filter first
mt_meta = mt_meta.filter_rows(mt_meta.hq_variant)

# Subset to significant genes and phenotypes
mt_meta = mt_meta.filter_rows(hl.literal(set(gene_ids)).contains(mt_meta.gene_id))
mt_meta = mt_meta.filter_cols(hl.literal(set(phenonames)).contains(mt_meta.phenoname))

out = f'{OUT_PATH}/variant_subset_META.mt'
mt_meta.write(out, overwrite=True)
print(f"  Written: {out}")

# Convert hail MT to parquet long appv table

In [ ]:
OUT_PATH = os.environ["AOU_GYM_DATA_ROOT"].rstrip("/") + "/allxall_exome_sumstats"
mt = hl.read_matrix_table(f"{OUT_PATH}/variant_subset_META.mt")

categories = list(mt.aggregate_cols(hl.agg.collect_as_set(mt.category)))
print("Categories:", categories)

for cat in categories:
    safe_cat = cat.replace('/', '_').replace(' ', '_')   # ← moved up
    print(f"\n--- {cat} ---")

    mt_cat = mt.filter_cols(mt.category == cat)
    ht = mt_cat.entries()
    ht = ht.annotate(
        id=hl.delimit([
            ht.locus.contig, hl.str(ht.locus.position),
            ht.alleles[0], ht.alleles[1]
        ], ':')
    )
    ht = ht.key_by()
    ht = ht.select(
        'id',
        gene_id     = ht.gene_id,
        gene_symbol = ht.gene_symbol,
        annotation  = ht.annotation,
        category    = ht.category,
        trait_type  = ht.trait_type,
        n_cases     = ht.n_cases,
        n_controls  = ht.n_controls,
        phenotype   = ht.phenoname,
        AC          = ht.AC[1],
        AN          = ht.AN,
        BETA        = ht.BETA,
        SE          = ht.SE,
        Pvalue      = ht.Pvalue,
        Pvalue_het  = ht.Pvalue_het,
        N           = ht.N,
        N_pops      = ht.N_pops,
    )

    out = f"{OUT_PATH}/variant_sumstats_associations_META/appv_{safe_cat}.parquet"
    ht.to_spark().write.mode("overwrite").parquet(out)
    print(f"  Written: {out}")
    
#     safe_cat = cat.replace('/', '_').replace(' ', '_')
#     local_out = f'<LOCAL_TEMP_DIR>/variant_sumstats_{safe_cat}.parquet'

#     df = pl.from_pandas(ht.to_pandas())
#     df.write_parquet(local_out)
#     subprocess.run(['gsutil', 'cp', local_out,
#         f'{OUT_PATH}/variant_sumstats_associations_META/appv_{safe_cat}.parquet'], check=True)
#     print(f"  Written: {safe_cat} ({df.shape})")

# Extract Variant metadata

In [ ]:
OUT_PATH = os.environ["AOU_GYM_DATA_ROOT"].rstrip("/") + "/allxall_exome_sumstats"

# gene_list = list(gt_df.filter(pl.col('Pvalue_FDR')<0.05)['gene_id'].unique())
# print("Number of unique genes:", len(gene_list))

ht = hl.read_table('gs://vwb-aou-allxall/v8/utility_ht/aou_exome_variant_qc_annotated.ht')

# Filter to your genes (top-level gene_id — easy)
# ht = ht.filter(hl.literal(set(gene_list)).contains(ht.gene_id))

wc = ht.vep.worst_csq_by_gene_canonical   # gene-specific, canonical/MANE transcript

ht = ht.annotate(
    variant_id = hl.delimit([
        ht.locus.contig, hl.str(ht.locus.position),
        ht.alleles[0], ht.alleles[1]
    ], ':'),
    chrom = ht.locus.contig,
    pos   = ht.locus.position,
    ref   = ht.alleles[0],
    alt   = ht.alleles[1],
    # gene-specific canonical consequence
    vep_consequence     = wc.most_severe_consequence,
    consequence_terms   = hl.delimit(wc.consequence_terms, ','),
    impact              = wc.impact,
    transcript_id       = wc.transcript_id,
    mane_select         = wc.mane_select,
    canonical           = wc.canonical,
    lof                 = wc.lof,
    lof_filter          = wc.lof_filter,
    lof_flags           = wc.lof_flags,
    polyphen_prediction = wc.polyphen_prediction,
    polyphen_score      = wc.polyphen_score,
    sift_prediction     = wc.sift_prediction,
    sift_score          = wc.sift_score,
    amino_acids         = wc.amino_acids,
    codons              = wc.codons,
    hgvsc               = wc.hgvsc,
    hgvsp               = wc.hgvsp,
    biotype             = wc.biotype,
    protein_id          = wc.protein_id,
    cds_start           = wc.cds_start,
    cds_end             = wc.cds_end,
    protein_start       = wc.protein_start,
    protein_end         = wc.protein_end,
    csq_score           = wc.csq_score,
    # per-ancestry allele frequencies (alt allele = index 1)
    AF_AFR = ht.freq.AFR.AF[1],
    AF_AMR = ht.freq.AMR.AF[1],
    AF_EAS = ht.freq.EAS.AF[1],
    AF_EUR = ht.freq.EUR.AF[1],
    AF_MID = ht.freq.MID.AF[1],
    AF_SAS = ht.freq.SAS.AF[1],
    AF_ALL = ht.freq.ALL.AF[1],
    AC_ALL = ht.freq.ALL.AC[1],
)

ht = ht.key_by()
ht = ht.select(
    'variant_id', 'chrom', 'pos', 'ref', 'alt',
    'gene_id', 'gene_symbol', 'annotation',
    'vep_consequence', 'consequence_terms', 'impact',
    'transcript_id', 'mane_select', 'canonical',
    'lof', 'lof_filter', 'lof_flags',
    'polyphen_prediction', 'polyphen_score',
    'sift_prediction', 'sift_score',
    'amino_acids', 'codons', 'hgvsp', 'hgvsc',
    'biotype', 'protein_start', 'protein_id',
    'AF_AFR', 'AF_AMR', 'AF_EAS', 'AF_EUR', 'AF_MID', 'AF_SAS', 'AF_ALL', 'AC_ALL',
)

# Write distributedly to GCS
ht.to_spark().write.mode('overwrite').parquet(f'{OUT_PATH}/aou_exome_variant_qc_annotated.parquet')

In [ ]:
# !gsutil cp <LOCAL_TEMP_DIR>/variant_annotations_exome_all.parquet <AOU_OUTPUT_ROOT>/ukbgym_exome_sumstats/variant_annotations_exome_all.parquet
